# 04. Pembobotan Kata dengan TF-IDF

Notebook ini khusus untuk proses **Term Frequency - Inverse Document Frequency (TF-IDF)** menggunakan hasil teks bersih dari `dataset_preprocessing.csv`.

### Konsep TF-IDF:
- **TF (Term Frequency)**: Mengukur seberapa sering kata muncul dalam suatu dokumen.
- **IDF (Inverse Document Frequency)**: Mengukur seberapa penting atau langkanya kata tersebut pada seluruh kumpulan dokumen training:
  $$\text{IDF}(t) = \log\left(\frac{1 + N}{1 + \text{DF}(t)}\right) + 1$$
- **TF-IDF**: $\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$, dinormalisasi dengan L2-norm.

## 1. Import Library

In [8]:
import numpy as np
import pandas as pd
from collections import Counter
from IPython.display import display

## 2. Membaca Dataset Preprocessing

In [9]:
df = pd.read_csv("dataset_preprocessing.csv")
print("Jumlah dokumen:", len(df))

# Tokenisasi per dokumen
dokumen_tokens = [str(text).split() for text in df["berita_clean"]]

Jumlah dokumen: 200


## 3. Pembagian Data Training (160) dan Testing (40)
Vocabulary dan nilai IDF dibangun dari data training (80 sport dan 80 finance) agar evaluasi data testing objektif.

In [10]:
indeks_train = (
    df.index[df["label"] == "sport"][:80].tolist() +
    df.index[df["label"] == "finance"][:80].tolist()
)
indeks_test = [idx for idx in df.index if idx not in indeks_train]

print(f"Jumlah Training: {len(indeks_train)} artikel")
print(f"Jumlah Testing : {len(indeks_test)} artikel")

Jumlah Training: 160 artikel
Jumlah Testing : 40 artikel


## 4. Pembentukan Vocabulary dan Nilai IDF

In [11]:
# Kosakata unik dari data training
vocabulary = sorted({
    kata for idx in indeks_train
    for kata in dokumen_tokens[idx]
    if kata != "label"
})

kata_to_idx = {kata: i for i, kata in enumerate(vocabulary)}

N = len(indeks_train)

# Hitung Document Frequency (DF)
df_count = np.zeros(len(vocabulary))

for idx in indeks_train:
    kata_unik_dokumen = set(dokumen_tokens[idx])

    for kata in kata_unik_dokumen:
        if kata in kata_to_idx:
            df_count[kata_to_idx[kata]] += 1

# Hitung IDF
idf = np.log((1 + N) / (1 + df_count)) + 1

print("Jumlah Vocabulary (Fitur Kata):", len(vocabulary))
print("Contoh 10 kata pertama:", vocabulary[:10])

Jumlah Vocabulary (Fitur Kata): 6269
Contoh 10 kata pertama: ['aaa', 'aau', 'abadi', 'abdullah', 'abidin', 'abimanyu', 'abraham', 'absen', 'absennya', 'academy']


## 5. Menghitung Matriks TF-IDF untuk Seluruh Dokumen
Menghitung bobot TF-IDF per dokumen dan melakukan normalisasi vektor (L2 Norm).

In [12]:
def hitung_vektor_tfidf(tokens):
    counts = Counter(tokens)
    total_kata = len(tokens)
    vektor = np.zeros(len(vocabulary))
    
    if total_kata > 0:
        for kata, freq in counts.items():
            if kata in kata_to_idx:
                posisi = kata_to_idx[kata]
                tf = freq / total_kata
                vektor[posisi] = tf * idf[posisi]
                
    norm = np.linalg.norm(vektor)
    return vektor / norm if norm > 0 else vektor

# Bentuk matriks TF-IDF (200 x V)
matriks_tfidf = np.vstack([hitung_vektor_tfidf(tokens) for tokens in dokumen_tokens])
print("Bentuk Matriks TF-IDF:", matriks_tfidf.shape)

Bentuk Matriks TF-IDF: (200, 6269)


## 6. Menyusun DataFrame TF-IDF & Menyimpan Hasil
Label diubah ke numerik (`sport = 1`, `finance = 2`) dan disimpan ke `hasil_tfidf.csv`.

In [13]:
print("label ada di vocabulary:", "label" in vocabulary)
print("id ada di vocabulary:", "id" in vocabulary)
print("Jumlah vocabulary:", len(vocabulary))

label ada di vocabulary: False
id ada di vocabulary: False
Jumlah vocabulary: 6269


In [14]:
df_tfidf = pd.DataFrame(matriks_tfidf, columns=vocabulary)
df_tfidf.insert(0, "label", df["label"].map({"sport": 1, "finance": 2}))
df_tfidf.insert(0, "id", df["id"])

df_tfidf.to_csv("hasil_tfidf.csv", index=False)

print("Data TF-IDF berhasil disimpan ke 'hasil_tfidf.csv'.")
print("Ukuran DataFrame:", df_tfidf.shape)

display(df_tfidf.iloc[:5, :12])

Data TF-IDF berhasil disimpan ke 'hasil_tfidf.csv'.
Ukuran DataFrame: (200, 6271)


,id,label,aaa,aau,abadi,abdullah,abidin,abimanyu,abraham,absen,absennya,academy
0,SP001,1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,SP002,1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,SP003,1,0.0,0.0,0.0,0.059515,0.0,0.0,0.0,0.0,0.0,0.0
3,SP004,1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,SP005,1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
